### 02 - MIA Feature Extraction

Builds the feature datasets for the attack classifier in notebook 03.

For each pair in each of the four datasets (target_train, target_test, shadow_train, shadow_test), runs the appropriate model and records the predicted probability and the per-pair loss. Each row is labelled `member=1` if the pair was in the model's training set, `member=0` if not.

Also reports the train-vs-test classifier gap for both models. The gap is the upper bound on attack effectiveness.

##### Inputs
- `outputs/models/{target,shadow}_{encoder,clf}.h5`
- `data/external/clinicalbert/*.npy`

##### Outputs
- `outputs/results/mia_features_shadow.csv` (attack training data, attacker has these labels)
- `outputs/results/mia_features_target.csv` (attack evaluation data, labels withheld in a real attack)
- `outputs/figures/mia_score_distributions.png`

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

In [ ]:

target_encoder = load_model("outputs/models/target_encoder.h5", compile=False)
target_clf     = load_model("outputs/models/target_clf.h5",     compile=False)
shadow_encoder = load_model("outputs/models/shadow_encoder.h5", compile=False)
shadow_clf     = load_model("outputs/models/shadow_clf.h5",     compile=False)

print("All four models loaded.")
print(f"  target encoder input shape:  {target_encoder.input_shape}")
print(f"  target clf input shape:      {target_clf.input_shape}")

In [ ]:
# Target side
x1_target_train = np.load("data/external/clinicalbert/x1_target_train.npy")
x2_target_train = np.load("data/external/clinicalbert/x2_target_train.npy")
y_target_train  = np.load("data/external/clinicalbert/y_target_train.npy")
x1_target_test  = np.load("data/external/clinicalbert/x1_target_test.npy")
x2_target_test  = np.load("data/external/clinicalbert/x2_target_test.npy")
y_target_test   = np.load("data/external/clinicalbert/y_target_test.npy")

# Shadow side
x1_shadow_train = np.load("data/external/clinicalbert/x1_shadow_train.npy")
x2_shadow_train = np.load("data/external/clinicalbert/x2_shadow_train.npy")
y_shadow_train  = np.load("data/external/clinicalbert/y_shadow_train.npy")
x1_shadow_test  = np.load("data/external/clinicalbert/x1_shadow_test.npy")
x2_shadow_test  = np.load("data/external/clinicalbert/x2_shadow_test.npy")
y_shadow_test   = np.load("data/external/clinicalbert/y_shadow_test.npy")

print("Embeddings loaded:")
print(f"  target train: {x1_target_train.shape[0]} pairs")
print(f"  target test:  {x1_target_test.shape[0]} pairs")
print(f"  shadow train: {x1_shadow_train.shape[0]} pairs")
print(f"  shadow test:  {x1_shadow_test.shape[0]} pairs")

In [ ]:
# Run each model on its own data and record the predicted match probability per pair.
# This is the raw signal the attack classifier will learn from.

def score_pairs(encoder, clf, x1, x2):
    """Run the full SNN+MLP pipeline on a pair set, return sigmoid probabilities.

    This is exactly what the original notebook's evaluation does:
      encode both sides, take absolute difference, push through MLP head.
    """
    enc1 = encoder.predict(x1, verbose=0)
    enc2 = encoder.predict(x2, verbose=0)
    diff = np.abs(enc1 - enc2)
    return clf.predict(diff, verbose=0).flatten()


# Score every (model, dataset) combination we care about.
# 'member=1' means the pair was in that model's training set.
probs_target_train = score_pairs(target_encoder, target_clf, x1_target_train, x2_target_train)
probs_target_test  = score_pairs(target_encoder, target_clf, x1_target_test,  x2_target_test)
probs_shadow_train = score_pairs(shadow_encoder, shadow_clf, x1_shadow_train, x2_shadow_train)
probs_shadow_test  = score_pairs(shadow_encoder, shadow_clf, x1_shadow_test,  x2_shadow_test)

print("Scoring complete.")
print(f"  probs_target_train: shape {probs_target_train.shape}, mean {probs_target_train.mean():.3f}")
print(f"  probs_target_test:  shape {probs_target_test.shape},  mean {probs_target_test.mean():.3f}")
print(f"  probs_shadow_train: shape {probs_shadow_train.shape}, mean {probs_shadow_train.mean():.3f}")
print(f"  probs_shadow_test:  shape {probs_shadow_test.shape},  mean {probs_shadow_test.mean():.3f}")

In [ ]:
# Build the feature tables. Each row is one pair with its predicted probability,
# binary cross-entropy loss, true label, and member flag (1=in training set, 0=not).
# The shadow table is what the attack classifier trains on; the target table is
# what we evaluate it against.

def per_pair_loss(probs, y_true, eps=1e-7):
    """Binary cross-entropy loss per pair. Equivalent to -log(p) when y=1, -log(1-p) when y=0."""
    p = np.clip(probs, eps, 1 - eps)
    return -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))


def build_feature_frame(probs, y_true, member_label, source_name):
    """One row per pair, two features (prob, loss) plus member label and provenance."""
    return pd.DataFrame({
        "prob":   probs,
        "loss":   per_pair_loss(probs, y_true),
        "y_true": y_true,
        "member": member_label,
        "source": source_name,
    })


# Shadow side - the attacker's training data for the attack classifier
shadow_features = pd.concat([
    build_feature_frame(probs_shadow_train, y_shadow_train, member_label=1, source_name="shadow_train"),
    build_feature_frame(probs_shadow_test,  y_shadow_test,  member_label=0, source_name="shadow_test"),
], ignore_index=True)

# Target side - the evaluation data
target_features = pd.concat([
    build_feature_frame(probs_target_train, y_target_train, member_label=1, source_name="target_train"),
    build_feature_frame(probs_target_test,  y_target_test,  member_label=0, source_name="target_test"),
], ignore_index=True)

print(f"Shadow features: {len(shadow_features)} rows")
print(f"Target features: {len(target_features)} rows")
print()
print("Sample of shadow_features:")
print(shadow_features.head())
print()
print("Mean loss by membership (shadow):")
print(shadow_features.groupby("member")["loss"].agg(["mean", "std"]))
print()
print("Mean loss by membership (target):")
print(target_features.groupby("member")["loss"].agg(["mean", "std"]))

In [ ]:
shadow_features.to_csv("outputs/results/mia_features_shadow.csv", index=False)
target_features.to_csv("outputs/results/mia_features_target.csv", index=False)

print("Saved:")
print("  outputs/results/mia_features_shadow.csv")
print("  outputs/results/mia_features_target.csv")